# Spark-Based MovieLens Recommender System

This notebook adapts my previous MovieLens recommendation system to Apache Spark.

The previous project used a single-machine Python workflow with the `surprise` library, including matrix factorization through SVD. This project keeps the same sampled MovieLens data and compares that previous-style implementation against Spark MLlib's ALS recommender.

## Setup

Set the local Python and Spark environment.

In [1]:
import os
import sys

os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(sys.executable)
print(sys.version)

C:\Users\ganzs\Documents\CUNY_Assignments\612\Project_5\.venv\Scripts\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


Quick check that Spark starts correctly.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("test") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

df = spark.createDataFrame([(1, "a"), (2, "b")], ["id", "val"])
df.show()

spark.stop()

4.1.2
+---+---+
| id|val|
+---+---+
|  1|  a|
|  2|  b|
+---+---+



In [3]:
import time
import warnings
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

## Load Sampled MovieLens Data

Load the same sampled MovieLens files used in the earlier project.

In [4]:
movies_sample = pd.read_parquet(
    "https://github.com/Siganz/CUNY_Assignments/raw/refs/heads/main/612/Project_4/movies_sample.parquet"
)

ratings_sample = pd.read_parquet(
    "https://github.com/Siganz/CUNY_Assignments/raw/refs/heads/main/612/Project_4/ratings_sample.parquet"
)

links_sample = pd.read_parquet(
    "https://github.com/Siganz/CUNY_Assignments/raw/refs/heads/main/612/Project_4/links_sample.parquet"
)

movies_sample.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,5,Father of the Bride Part II (1995),Comedy
4,6,Heat (1995),Action|Crime|Thriller


In [5]:
ratings_sample.head()

,userId,movieId,rating,timestamp
0,3,151,5.0,1084485165
1,3,153,3.0,1084484623
2,3,158,3.0,1084484425
3,10,593,5.0,1169262675
4,10,1092,3.0,1169264878


## Basic Dataset Summary

General statistics about the sample data.

In [6]:
n_ratings = len(ratings_sample)
n_users = ratings_sample["userId"].nunique()
n_movies = ratings_sample["movieId"].nunique()
possible_ratings = n_users * n_movies
density = n_ratings / possible_ratings

summary = pd.DataFrame({
    "Metric": [
        "Number of ratings",
        "Number of users",
        "Number of rated movies",
        "Possible user-movie pairs",
        "Rating matrix density"
    ],
    "Value": [
        n_ratings,
        n_users,
        n_movies,
        possible_ratings,
        density
    ]
})

summary

,Metric,Value
0,Number of ratings,3.886980e+05
1,Number of users,6.598500e+04
2,Number of rated movies,1.000000e+03
3,Possible user-movie pairs,6.598500e+07
4,Rating matrix density,5.890702e-03


In [7]:
ratings_sample["rating"].describe()

count    388698.000000
mean          3.648711
std           1.006105
min           0.500000
25%           3.000000
50%           4.000000
75%           4.500000
max           5.000000
Name: rating, dtype: float64

In [8]:
ratings_sample["rating"].value_counts().sort_index()

rating
0.5      4719
1.0      8358
1.5      5302
2.0     20448
2.5     18403
3.0     67675
3.5     53709
4.0    108007
4.5     41494
5.0     60583
Name: count, dtype: int64

## Train/Test Split

Use the same split for both models.


In [9]:
train_pd, test_pd = train_test_split(
    ratings_sample,
    test_size=0.2,
    random_state=42
)

print(f"Training rows: {len(train_pd):,}")
print(f"Testing rows: {len(test_pd):,}")

Training rows: 310,958
Testing rows: 77,740


## Baseline Model: Surprise SVD

Train a tuned Surprise SVD model and evaluate it with RMSE.

In [10]:
from surprise import Dataset, Reader, SVD, accuracy

reader = Reader(rating_scale=(0.5, 5.0))

train_data = Dataset.load_from_df(
    train_pd[["userId", "movieId", "rating"]],
    reader
)

trainset = train_data.build_full_trainset()

testset = list(
    test_pd[["userId", "movieId", "rating"]]
    .itertuples(index=False, name=None)
)

# Parameters from previous assignment's grid search.
svd = SVD(
    n_factors=20,
    n_epochs=20,
    lr_all=0.01,
    reg_all=0.05,
    random_state=42
)

start = time.time()
svd.fit(trainset)
svd_predictions = svd.test(testset)
svd_runtime = time.time() - start

svd_rmse = accuracy.rmse(svd_predictions)

print(f"Tuned Surprise SVD RMSE: {svd_rmse:.4f}")
print(f"Tuned Surprise SVD runtime: {svd_runtime:.2f} seconds")

RMSE: 0.8863
Tuned Surprise SVD RMSE: 0.8863
Tuned Surprise SVD runtime: 1.73 seconds


## Spark Model: ALS

Train a Spark ALS model on the same train/test split.

This runs Spark locally, not on a cluster.

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .appName("MovieLens_ALS_Recommender")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.executorEnv.PYSPARK_PYTHON", sys.executable)
    .config("spark.python.worker.reuse", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

In [12]:
train_spark = spark.createDataFrame(train_pd).select(
    col("userId").cast("int"),
    col("movieId").cast("int"),
    col("rating").cast("float")
).cache()

test_spark = spark.createDataFrame(test_pd).select(
    col("userId").cast("int"),
    col("movieId").cast("int"),
    col("rating").cast("float")
).cache()

print(f"Spark training rows: {train_spark.count():,}")
print(f"Spark testing rows: {test_spark.count():,}")

Spark training rows: 310,958
Spark testing rows: 77,740


In [13]:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=20,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop",
    nonnegative=True,
    seed=42
)

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

start = time.time()

als_model = als.fit(train_spark)
als_predictions = als_model.transform(test_spark)
als_rmse = evaluator.evaluate(als_predictions)

als_runtime = time.time() - start

print(f"Spark ALS RMSE: {als_rmse:.4f}")
print(f"Spark ALS runtime: {als_runtime:.2f} seconds")

Spark ALS RMSE: 1.2301
Spark ALS runtime: 7.08 seconds


## Compare SVD and Spark ALS

SVD and ALS are not identical algorithms, but both are matrix factorization methods for collaborative filtering, so this is a reasonable comparison.


In [14]:
comparison = pd.DataFrame({
    "Model": ["Surprise SVD", "Spark ALS"],
    "Implementation": ["Single-machine Python", "PySpark local mode"],
    "Algorithm family": ["Matrix factorization", "Matrix factorization"],
    "RMSE": [svd_rmse, als_rmse],
    "Runtime_seconds": [svd_runtime, als_runtime]
})

comparison

,Model,Implementation,Algorithm family,RMSE,Runtime_seconds
0,Surprise SVD,Single-machine Python,Matrix factorization,0.886340,1.733460
1,Spark ALS,PySpark local mode,Matrix factorization,1.230123,7.080606


In [15]:
comparison.set_index("Model")[["RMSE", "Runtime_seconds"]]

,RMSE,Runtime_seconds
Model,,
Surprise SVD,0.886340,1.733460
Spark ALS,1.230123,7.080606


On this sampled MovieLens data, Surprise SVD had better RMSE and lower runtime than Spark ALS.

Spark was not better here because the dataset is small and Spark adds setup overhead. The SVD model is also simpler in this notebook because it reuses hyperparameters from the earlier grid search instead of repeating the full item-item comparison.

Spark would be more interesting to test on a larger, less filtered MovieLens dataset with millions of ratings.

## Generate Sample Recommendations

Create top movie recommendations for a few users and attach movie titles.

In [16]:
from pyspark.sql.functions import explode

user_recs = als_model.recommendForAllUsers(5)

recs_exploded = (
    user_recs
    .select("userId", explode("recommendations").alias("rec"))
    .select(
        col("userId"),
        col("rec.movieId").alias("movieId"),
        col("rec.rating").alias("predicted_rating")
    )
)

movies_spark = spark.createDataFrame(movies_sample).select(
    col("movieId").cast("int"),
    col("title"),
    col("genres")
)

sample_recs = (
    recs_exploded
    .join(movies_spark, on="movieId", how="left")
    .orderBy("userId", col("predicted_rating").desc())
)

sample_recs.show(20, truncate=False)

+-------+------+----------------+-------------------------------------------------------+-----------------------------------------------+
|movieId|userId|predicted_rating|title                                                  |genres                                         |
+-------+------+----------------+-------------------------------------------------------+-----------------------------------------------+
|151    |3     |4.711111        |Rob Roy (1995)                                         |Action|Drama|Romance|War                       |
|1358   |3     |4.6023026       |Sling Blade (1996)                                     |Drama                                          |
|3105   |3     |4.513012        |Awakenings (1990)                                      |Drama|Mystery                                  |
|1212   |3     |4.492548        |Third Man, The (1949)                                  |Film-Noir|Mystery|Thriller                     |
|923    |3     |4.446497        |C

### Spark ALS Recommendations for Specific Users:

In [17]:
users_to_show = [3, 10, 16]

# Show recommendations only for the selected users. 
sample_recs_selected = (
    sample_recs
    .filter(col("userId").isin(users_to_show))
    .orderBy("userId", col("predicted_rating").desc())
)

sample_recs_selected.show(15, truncate=False)

+-------+------+----------------+--------------------------------------------+--------------------------------+
|movieId|userId|predicted_rating|title                                       |genres                          |
+-------+------+----------------+--------------------------------------------+--------------------------------+
|151    |3     |4.711111        |Rob Roy (1995)                              |Action|Drama|Romance|War        |
|1358   |3     |4.6023026       |Sling Blade (1996)                          |Drama                           |
|3105   |3     |4.513012        |Awakenings (1990)                           |Drama|Mystery                   |
|1212   |3     |4.492548        |Third Man, The (1949)                       |Film-Noir|Mystery|Thriller      |
|923    |3     |4.446497        |Citizen Kane (1941)                         |Drama|Mystery                   |
|593    |10    |4.1558566       |Silence of the Lambs, The (1991)            |Crime|Horror|Thriller     

### Original Movie Ratings From Users:

In [18]:
users_to_check = [3, 10, 16]

# Pull user's actual ratings for comparison. 
actual_ratings_pd = (
    ratings_sample[ratings_sample["userId"].isin(users_to_check)]
    .merge(movies_sample, on="movieId", how="left")
    [["userId", "movieId", "title", "genres", "rating"]]
    .sort_values(["userId", "rating", "title"], ascending=[True, False, True])
)

actual_ratings_pd.head(100)

,userId,movieId,title,genres,rating
0,3,151,Rob Roy (1995),Action|Drama|Romance|War,5.0
1,3,153,Batman Forever (1995),Action|Adventure|Comedy|Crime,3.0
2,3,158,Casper (1995),Adventure|Children,3.0
3,10,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,5.0
7,10,2959,Fight Club (1999),Action|Crime|Drama|Thriller,4.0
12,10,168252,Logan (2017),Action|Sci-Fi,4.0
8,10,3793,X-Men (2000),Action|Adventure|Sci-Fi,3.5
4,10,1092,Basic Instinct (1992),Crime|Mystery|Thriller,3.0
11,10,64957,"Curious Case of Benjamin Button, The (2008)",Drama|Fantasy|Mystery|Romance,3.0
10,10,54259,Stardust (2007),Adventure|Comedy|Fantasy|Romance,3.0


Spark ALS partially matched the users’ observed ratings, but the output needs filtering before it can be treated as true recommendations. Spark does not automatically remove movies users have already rated, so items like Rob Roy for user 3 and Silence of the Lambs for user 10 appeared because the model predicted them highly from known ratings. User 16’s results were weaker, with action/sci-fi recommendations despite low ratings for similar movies. Overall, the ALS output shows some learned preference patterns, but already-rated movies should be removed before evaluating recommendation quality.

### Surprise SVD Recommendations for Specific Users:

In [19]:
users_to_show = [3, 10, 16]
n_recs = 5

known_movie_ids = [
    trainset.to_raw_iid(inner_iid)
    for inner_iid in trainset.all_items()
]

recommendation_rows = []

for user_id in users_to_show:
    predictions = [
        svd.predict(user_id, movie_id)
        for movie_id in known_movie_ids
    ]

    top_predictions = sorted(
        predictions,
        key=lambda x: x.est,
        reverse=True
    )[:n_recs]

    for pred in top_predictions:
        recommendation_rows.append({
            "userId": pred.uid,
            "movieId": pred.iid,
            "predicted_rating": pred.est
        })

surprise_recs_raw = pd.DataFrame(recommendation_rows)

surprise_recs_raw = (
    surprise_recs_raw
    .merge(movies_sample, on="movieId", how="left")
    [["userId", "movieId", "title", "genres", "predicted_rating"]]
    .sort_values(["userId", "predicted_rating"], ascending=[True, False])
    .reset_index(drop=True)
)

surprise_recs_raw

,userId,movieId,title,genres,predicted_rating
0,3,2019,Seven Samurai (Shichinin no samurai) (1954),Action|Adventure|Drama,4.659706
1,3,202439,Parasite (2019),Comedy|Drama,4.639841
2,3,5971,My Neighbor Totoro (Tonari no Totoro) (1988),Animation|Children|Drama|Fantasy,4.620868
3,3,318,"Shawshank Redemption, The (1994)",Crime|Drama,4.616891
4,3,56782,There Will Be Blood (2007),Drama|Western,4.580476
5,10,2019,Seven Samurai (Shichinin no samurai) (1954),Action|Adventure|Drama,3.887512
6,10,5971,My Neighbor Totoro (Tonari no Totoro) (1988),Animation|Children|Drama|Fantasy,3.870091
7,10,720,Wallace & Gromit: The Best of Aardman Animatio...,Adventure|Animation|Comedy,3.865435
8,10,318,"Shawshank Redemption, The (1994)",Crime|Drama,3.856201
9,10,2324,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,3.846177


The Surprise SVD recommendations look stronger than the initial Spark ALS results. These recommendations were filtered to remove movies users had already rated, so they are closer to true recommendations.

The titles also seem more reasonable overall. SVD recommends strong options like Seven Samurai, Parasite, The Shawshank Redemption, and 12 Angry Men. This suggests the model may be producing better recommendations, although it may also be favoring broadly popular or highly rated movies.

## Conclusion

The Spark version adds more setup because it requires a Spark session, Spark DataFrames, and extra configuration. However, the SVD version here is simpler than the previous assignment because it reuses the tuned hyperparameters from the earlier grid search and does not repeat the full item-item comparison.

On this sampled dataset, Spark ALS was not faster or more accurate than Surprise SVD. Spark may be more useful on a larger, less filtered MovieLens dataset with millions of ratings, where the user-item matrix is sparse and harder to handle locally.

For this sample, Surprise SVD was the better fit. Spark ALS is mainly useful here as a scalable version to test on larger data.

In [20]:
spark.stop()